In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    KFold,
    cross_validate,
    cross_val_score,
)
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    recall_score,
    f1_score,
    make_scorer,
)
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

In [4]:
# === CASE 1: AdaBoost on the Diagnosis of Diabetes (classification) ===

# 1. Load dataset
df1 = pd.read_csv("diagnosed_diabetes for classification.csv")

print("Case 1 columns:", df1.columns.tolist())

target_col1 = "diagnosed_diabetes"

# 2. Separate features and target
X1 = df1.drop(columns=[target_col1])
y1 = df1[target_col1]

# 3. Train–test split
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=0, stratify=y1
)

# 4. AdaBoost classifier (decision stumps)
base_clf1 = DecisionTreeClassifier(max_depth=1, random_state=0)

ada_clf1 = AdaBoostClassifier(
    estimator=base_clf1,
    n_estimators=200,
    learning_rate=0.1,
    random_state=0,
)

# 5. Train on train set
ada_clf1.fit(X1_train, y1_train)

# 6. Predict on test set
y1_pred = ada_clf1.predict(X1_test)

# 7. Confusion matrix & classification report
print("\n=== CASE 1: Diagnosed Diabetes (AdaBoost) ===")
cm1 = confusion_matrix(y1_test, y1_pred)
print("Confusion matrix:\n", cm1)
print("\nClassification report:\n", classification_report(y1_test, y1_pred))

# 8. Cross-validation on full dataset (5-fold stratified)
print("\n--- CASE 1: 5-fold Cross-Validation (AdaBoost) ---")

cv1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

scoring1 = {
    "accuracy": "accuracy",
    "recall_1": make_scorer(recall_score, pos_label=1),
    "f1_1": make_scorer(f1_score, pos_label=1),
}

cv_results1 = cross_validate(
    ada_clf1, X1, y1, cv=cv1, scoring=scoring1, n_jobs=-1
)

for metric in scoring1.keys():
    vals = cv_results1[f"test_{metric}"]
    print(f"{metric}: {vals.mean():.3f} ± {vals.std():.3f}")

Case 1 columns: ['age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'gender_Male', 'gender_Other', 'ethnicity_Black', 'ethnicity_Hispanic', 'ethnicity_Other', 'ethnicity_White', 'education_level_Highschool', 'education_level_No formal', 'education_level_Postgraduate', 'income_level_Low', 'income_level_Lower-Middle', 'income_level_Middle', 'income_level_Upper-Middle', 'employment_status_Retired', 'employment_status_Student', 'employment_status_Unemployed', 'smoking_status_Former', 'smoking_status_Never', 'diagnosed_diabetes']


/apps/python/3.10/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(



=== CASE 1: Diagnosed Diabetes (AdaBoost) ===
Confusion matrix:
 [[2621 5379]
 [2027 9973]]

Classification report:
               precision    recall  f1-score   support

           0       0.56      0.33      0.41      8000
           1       0.65      0.83      0.73     12000

    accuracy                           0.63     20000
   macro avg       0.61      0.58      0.57     20000
weighted avg       0.62      0.63      0.60     20000


--- CASE 1: 5-fold Cross-Validation (AdaBoost) ---


/apps/python/3.10/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/apps/python/3.10/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/apps/python/3.10/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/apps/python/3.10/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.wa

accuracy: 0.633 ± 0.003
recall_1: 0.837 ± 0.001
f1_1: 0.732 ± 0.002


In [6]:
# ============================================
# ANALYSIS FOR CASE 1 – Diagnosed Diabetes (0/1)
# ============================================

print("\n" + "="*70)
print("CASE 1 – ANALYSIS: Diagnosed Diabetes Classification (AdaBoost)")
print("="*70)

print("""
Test-set performance (single 80/20 split):
- Accuracy ≈ 0.63
- For class 1 (Diabetes):
    • Recall ≈ 0.83  → the model correctly catches about 83% of diabetics.
    • F1-score ≈ 0.73
- For class 0 (No Diabetes):
    • Recall ≈ 0.33  → the model only correctly identifies ~33% of healthy individuals.
    • Many healthy people are flagged as diabetic (false positives).

Cross-validation (5-fold):
- Accuracy ≈ 0.633 ± 0.003
- Recall for class 1 ≈ 0.837 ± 0.001
- F1-score for class 1 ≈ 0.732 ± 0.002

Interpretation:
- The dataset is mildly imbalanced (roughly 40% No Diabetes, 60% Diabetes),
  but not extremely skewed, so we did not apply SMOTE here.
- AdaBoost is very good at catching diabetics (high recall and F1 for class 1).
- It is much weaker at correctly identifying healthy individuals (low recall for class 0).
- The cross-validation results are very stable (small standard deviations),
  which suggests the model generalizes well and is not overly sensitive to how we split the data.
- Case 1 works as a non-invasive screening classifier that prioritizes not missing diabetics*:
""")


CASE 1 – ANALYSIS: Diagnosed Diabetes Classification (AdaBoost)

Test-set performance (single 80/20 split):
- Accuracy ≈ 0.63
- For class 1 (Diabetes):
    • Recall ≈ 0.83  → the model correctly catches about 83% of diabetics.
    • F1-score ≈ 0.73
- For class 0 (No Diabetes):
    • Recall ≈ 0.33  → the model only correctly identifies ~33% of healthy individuals.
    • Many healthy people are flagged as diabetic (false positives).

Cross-validation (5-fold):
- Accuracy ≈ 0.633 ± 0.003
- Recall for class 1 ≈ 0.837 ± 0.001
- F1-score for class 1 ≈ 0.732 ± 0.002

Interpretation:
- The dataset is mildly imbalanced (roughly 40% No Diabetes, 60% Diabetes),
  but not extremely skewed, so we did not apply SMOTE here.
- AdaBoost is very good at catching diabetics (high recall and F1 for class 1).
- It is much weaker at correctly identifying healthy individuals (low recall for class 0).
- The cross-validation results are very stable (small standard deviations),
  which suggests the model genera

In [7]:
# === CASE 2: AdaBoost on No vs Pre-diabetes(classification) ===

df2 = pd.read_csv("prediabetes_for_classification.csv")

print("\nCase 2 shape (raw):", df2.shape)
print("Case 2 columns:", df2.columns.tolist())

target2 = "diabetes_stage"

# 1. Inspect target
print("\nCase 2 unique values in diabetes_stage:")
print(df2[target2].unique())

print("\nCase 2 value counts:")
print(df2[target2].value_counts())

# 2. Drop rows with missing target (safety)
df2 = df2.dropna(subset=[target2])

# In your file, diabetes_stage is already coded as 0/1:
# 0.0 = No Diabetes, 1.0 = Pre-Diabetes
y2 = df2[target2].astype(int)
X2 = df2.drop(columns=[target2])

# 3. Train–test split (stratified)
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=0, stratify=y2
)

print("\nCase 2 class counts in train set:")
print(y2_train.value_counts())
print("\nCase 2 class counts in test set:")
print(y2_test.value_counts())

# 4. AdaBoost with a class-weighted base tree
#    This is a simple alternative to SMOTE in this environment.
base_clf2 = DecisionTreeClassifier(
    max_depth=1,
    random_state=0,
    class_weight="balanced"  # give more weight to minority class
)

ada_clf2 = AdaBoostClassifier(
    estimator=base_clf2,
    n_estimators=200,
    learning_rate=0.1,
    random_state=0,
)

# 5. Train
ada_clf2.fit(X2_train, y2_train)

# 6. Predict
y2_pred = ada_clf2.predict(X2_test)

# 7. Confusion matrix & report
print("\n=== CASE 2: No vs Pre-Diabetes (AdaBoost, class-weighted base tree) ===")
cm2 = confusion_matrix(y2_test, y2_pred)
print("Confusion matrix:\n", cm2)
print("\nClassification report:\n", classification_report(y2_test, y2_pred))

# 8. Cross-validation (5-fold stratified)
print("\n--- CASE 2: 5-fold Cross-Validation (AdaBoost) ---")

cv2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

scoring2 = {
    "accuracy": "accuracy",
    "recall_1": make_scorer(recall_score, pos_label=1),
    "f1_1": make_scorer(f1_score, pos_label=1),
}

cv_results2 = cross_validate(
    ada_clf2, X2, y2, cv=cv2, scoring=scoring2, n_jobs=-1
)

for metric in scoring2.keys():
    vals = cv_results2[f"test_{metric}"]
    print(f"{metric}: {vals.mean():.3f} ± {vals.std():.3f}")


Case 2 shape (raw): (39826, 33)
Case 2 columns: ['age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'diabetes_stage', 'gender_Male', 'gender_Other', 'ethnicity_Black', 'ethnicity_Hispanic', 'ethnicity_Other', 'ethnicity_White', 'education_level_Highschool', 'education_level_No formal', 'education_level_Postgraduate', 'income_level_Low', 'income_level_Lower-Middle', 'income_level_Middle', 'income_level_Upper-Middle', 'employment_status_Retired', 'employment_status_Student', 'employment_status_Unemployed', 'smoking_status_Former', 'smoking_status_Never']

Case 2 unique values in diabetes_stage:
[0. 1.]

Case 2 value counts:
diabetes_stage
1.0    31845
0.0     7981
Name: count, dtype: int64

Case 2 class counts in train set:
diabetes_stage
1    2547

/apps/python/3.10/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(



=== CASE 2: No vs Pre-Diabetes (AdaBoost, class-weighted base tree) ===
Confusion matrix:
 [[1029  567]
 [2812 3558]]

Classification report:
               precision    recall  f1-score   support

           0       0.27      0.64      0.38      1596
           1       0.86      0.56      0.68      6370

    accuracy                           0.58      7966
   macro avg       0.57      0.60      0.53      7966
weighted avg       0.74      0.58      0.62      7966


--- CASE 2: 5-fold Cross-Validation (AdaBoost) ---


/apps/python/3.10/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/apps/python/3.10/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/apps/python/3.10/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/apps/python/3.10/lib/python3.10/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.wa

accuracy: 0.576 ± 0.003
recall_1: 0.562 ± 0.005
f1_1: 0.679 ± 0.003


In [9]:
# ============================================
# ANALYSIS FOR CASE 2 – No Diabetes vs Pre-Diabetes (0/1)
# ============================================

print("\n" + "="*70)
print("CASE 2 – ANALYSIS: No Diabetes vs Pre-Diabetes Classification (AdaBoost)")
print("="*70)

print("""
Dataset characteristics:
- Moderately imbalanced: about 80% of samples are class 1 (Pre-Diabetes)
  and only 20% are class 0 (No Diabetes).

Single train–test evaluation:
- Overall accuracy ≈ 0.58

From the confusion matrix:
[[1029  567]
 [2812 3558]]

- For class 0 (No Diabetes):
    * True Negatives = 1029
    * False Positives = 567
    * Precision ≈ 0.27
    * Recall   ≈ 0.64
    * F1-score ≈ 0.38

- For class 1 (Pre-Diabetes):
    * False Negatives = 2812
    * True Positives = 3558
    * Precision ≈ 0.86
    * Recall   ≈ 0.56
    * F1-score ≈ 0.68

Interpretation of single split:
- The model now predicts BOTH classes (unlike Case 1 that predicted only class 1 for everyone).
- It is reasonably precise for Pre-Diabetes.
- However, recall for Pre-Diabetes (~0.56) means it still misses ~44% of the pre-diabetic cases.
- For No Diabetes, recall is ~0.64 but precision is low (~0.27), meaning many people predicted as 'No Diabetes' are actually pre-diabetic.

Cross-validation summary (5-fold CV on the full dataset):
- Accuracy  ≈ 0.576 ± 0.003
- Recall_1  ≈ 0.562 ± 0.005   (for class 1 = Pre-Diabetes)
- F1_1      ≈ 0.679 ± 0.003   (for class 1 = Pre-Diabetes)

Interpretation of cross-validation:
- Cross-validated metrics are very close to the single test-split results,
  which suggests the model is stable and not overly sensitive to how we
  split the data.
- F1 for the Pre-Diabetes class (~0.68) reflects a compromise between:
    → catching pre-diabetic cases (recall ≈ 0.56)
    → and keeping predictions reasonably precise (precision ≈ 0.86).
- It also highlights the impact of class imbalance and motivates future work:
    → trying SMOTE or other resampling methods,
    → tuning thresholds for different clinical priorities (e.g., very high sensitivity vs fewer false alarms),
    → or using more specialized models for imbalanced classification.
""")


CASE 2 – ANALYSIS: No Diabetes vs Pre-Diabetes Classification (AdaBoost)

Dataset characteristics:
- Moderately imbalanced: about 80% of samples are class 1 (Pre-Diabetes)
  and only 20% are class 0 (No Diabetes).

Single train–test evaluation:
- Overall accuracy ≈ 0.58

From the confusion matrix:
[[1029  567]
 [2812 3558]]

- For class 0 (No Diabetes):
    * True Negatives = 1029
    * False Positives = 567
    * Precision ≈ 0.27
    * Recall   ≈ 0.64
    * F1-score ≈ 0.38

- For class 1 (Pre-Diabetes):
    * False Negatives = 2812
    * True Positives = 3558
    * Precision ≈ 0.86
    * Recall   ≈ 0.56
    * F1-score ≈ 0.68

Interpretation of single split:
- The model now predicts BOTH classes (unlike Case 1 that predicted only class 1 for everyone).
- It is reasonably precise for Pre-Diabetes.
- However, recall for Pre-Diabetes (~0.56) means it still misses ~44% of the pre-diabetic cases.
- For No Diabetes, recall is ~0.64 but precision is low (~0.27), meaning many people predicted

In [10]:
# === CASE 3: AdaBoost on diabetes_risk_score (regression) ===

df3 = pd.read_csv("diabetes_preprocessed_for_risk_score.csv")

print("\nCase 3 columns:", df3.columns.tolist())

target_col3 = "diabetes_risk_score"

X3 = df3.drop(columns=[target_col3])
y3 = df3[target_col3]

# 1. Train–test split
X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=0
)

# 2. AdaBoost regressor
base_reg = DecisionTreeRegressor(max_depth=3, random_state=0)

ada_reg = AdaBoostRegressor(
    estimator=base_reg,
    n_estimators=200,
    learning_rate=0.1,
    random_state=0,
)

# 3. Train
ada_reg.fit(X3_train, y3_train)

# 4. Predict
y3_pred = ada_reg.predict(X3_test)

# 5. Metrics on test split
rmse = mean_squared_error(y3_test, y3_pred, squared=False)
mae = mean_absolute_error(y3_test, y3_pred)
r2 = r2_score(y3_test, y3_pred)

print("\n=== CASE 3: Risk Score (AdaBoostRegressor) ===")
print(f"RMSE: {rmse:.3f}")
print(f"MAE:  {mae:.3f}")
print(f"R²:   {r2:.3f}")

# 6. Cross-validation: R² across 5 folds
print("\n--- CASE 3: 5-fold Cross-Validation (AdaBoostRegressor) ---")

kf3 = KFold(n_splits=5, shuffle=True, random_state=0)

cv_r2 = cross_val_score(
    ada_reg, X3, y3, cv=kf3, scoring="r2", n_jobs=-1
)

print(f"R² (CV mean ± std): {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")


Case 3 columns: ['age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'diabetes_risk_score', 'gender_Male', 'gender_Other', 'ethnicity_Black', 'ethnicity_Hispanic', 'ethnicity_Other', 'ethnicity_White', 'education_level_Highschool', 'education_level_No formal', 'education_level_Postgraduate', 'income_level_Low', 'income_level_Lower-Middle', 'income_level_Middle', 'income_level_Upper-Middle', 'employment_status_Retired', 'employment_status_Student', 'employment_status_Unemployed', 'smoking_status_Former', 'smoking_status_Never']


/apps/python/3.10/lib/python3.10/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(



=== CASE 3: Risk Score (AdaBoostRegressor) ===
RMSE: 2.756
MAE:  2.173
R²:   0.908

--- CASE 3: 5-fold Cross-Validation (AdaBoostRegressor) ---
R² (CV mean ± std): 0.907 ± 0.001


In [11]:
# ============================================
# ANALYSIS FOR CASE 3 – Diabetes Risk Score (Regression)
# ============================================

print("\n" + "="*70)
print("CASE 3 – ANALYSIS: Diabetes Risk Score Regression (AdaBoostRegressor)")
print("="*70)

print("""
Interpretation:
- R² = 0.91 means the model explains about 91% of the variation in risk scores.
- Errors (2–3 points) are small compared to the full 0–100 range.
- This indicates VERY strong predictive ability.

Cross-validation (5-fold, full dataset):
- Mean R² ≈ 0.907 with a very small standard deviation (~0.001).
- This shows that the high performance is consistent across different folds

Overall meaning:
- AdaBoostRegressor is performing extremely well for this regression task.
- The model produces accurate and stable predictions of diabetes risk score
  using mostly non-invasive features.
- Among the three cases, Case 3 (regression on risk score) is the strongest model.
- It is a good candidate for a decision-support tool, where clinicians or
  public-health practitioners want a continuous risk estimate instead of
  a simple yes/no classification.
""")


CASE 3 – ANALYSIS: Diabetes Risk Score Regression (AdaBoostRegressor)

Interpretation:
- R² = 0.91 means the model explains about 91% of the variation in risk scores.
- Errors (2–3 points) are small compared to the full 0–100 range.
- This indicates VERY strong predictive ability.

Cross-validation (5-fold, full dataset):
- Mean R² ≈ 0.907 with a very small standard deviation (~0.001).
- This shows that the high performance is consistent across different folds

Overall meaning:
- AdaBoostRegressor is performing extremely well for this regression task.
- The model produces accurate and stable predictions of diabetes risk score
  using mostly non-invasive features.
- Among the three cases, Case 3 (regression on risk score) is the strongest model.
- It is a good candidate for a decision-support tool, where clinicians or
  public-health practitioners want a continuous risk estimate instead of
  a simple yes/no classification.

